In [1]:
!pip install "git+https://github.com/facebookresearch/detectron2.git"

  Cloning https://github.com/facebookresearch/detectron2.git to /tmp/pip-req-build-4vt0pzmp
  Running command git clone --filter=blob:none --quiet https://github.com/facebookresearch/detectron2.git /tmp/pip-req-build-4vt0pzmp
  Resolved https://github.com/facebookresearch/detectron2.git to commit fd27788985af0f4ca800bca563acdb700bb890e2
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 1.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 6.0 MB/s eta 0:00:00
  Created wheel for detectron2: filename=detectron2-0.6-cp312-cp312-linux_x86_64.whl size=7085017 sha256=e112fb0fb4d0fb26062f5f769718440d19e16d4d480536ba8ced80b17655ea69
  Stored in directory: /tmp/pip-ephem-wheel-cache-4wrox1c5/wheels/d3/6e/bd/1969578f1456a6be2d6f083da65c669f450b23b8f3d1ac14c1
  Created wheel for fvcore: filename=fvcore-0.1.5.post20221221-py3-none-any.whl size=61397 sha256=871d491db08489fc9dfa28317

In [2]:
"""
═══════════════════════════════════════════════════════════════════════════════
  BoneScan AI — BTXRD Test Split Evaluation (Metadata CSV)
  ─────────────────────────────────────────────────────────
  Images  : .../test/images
  Labels  : test_metadata.csv  →  class_label column
              "Normal"    →  gt_stage1=0, gt_pipe="Normal"
              "Benign"    →  gt_stage1=1, gt_pipe="Benign"
              "Malignant" →  gt_stage1=1, gt_pipe="Malignant"

  PIPELINE
  ─────────
  Stage 1 : Faster R-CNN  →  Normal (no ROI) vs Tumor (ROI found)
  Stage 2 : ConvNeXtTiny-SE on each ROI crop  →  Benign vs Malignant
            (only runs when Stage 1 fires)
═══════════════════════════════════════════════════════════════════════════════
"""

# ─── CELL 0 — Install detectron2 (run as separate cell first) ─────────────────
# import subprocess, sys
# subprocess.check_call([sys.executable, "-m", "pip", "install",
#     "git+https://github.com/facebookresearch/detectron2.git", "--quiet"])

# ─── 1. IMPORTS ───────────────────────────────────────────────────────────────
import warnings, json, time
from pathlib import Path
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T
from PIL import Image, ImageOps
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score,
    confusion_matrix, classification_report,
    matthews_corrcoef, cohen_kappa_score,
    roc_curve, precision_recall_curve,
    ConfusionMatrixDisplay,
)

try:
    from detectron2.config import get_cfg
    from detectron2 import model_zoo
    from detectron2.engine import DefaultPredictor
    D2_OK = True
    print("✅  detectron2 ready")
except ImportError:
    D2_OK = False
    print("❌  detectron2 not found")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅  Device : {DEVICE}")

# ─── 2. PATHS & SETTINGS ──────────────────────────────────────────────────────
ROI_MODEL_PATH = "/kaggle/input/models/samiulalim172/roimodel/pytorch/default/1/final_best.pth"
DET_MODEL_PATH = "/kaggle/input/models/samiulalim172/detmodel/pytorch/default/1/det_model_best.pth"
IMG_DIR        = Path("/kaggle/input/datasets/blaster21/btxrd-dataset/BTXRD/images")   # ← changed
META_CSV       = Path("/kaggle/input/datasets/samiulalim172/meta-new/test_metadata.csv")  # ← changed
OUTPUT_DIR     = Path("/kaggle/working/eval_btxrd_test")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

IMG_EXTS         = {".jpg", ".jpeg", ".png", ".bmp", ".tiff", ".webp"}
DET_SCORE_THRESH = 0.15
NMS_THRESH       = 0.40
CLS_THRESHOLD    = 0.50
ROI_PADDING      = 0.08
AGG_MODE         = "Any Malignant"   # Any Malignant | Majority Vote | Max Probability

# ─── 3. ARCHITECTURE ──────────────────────────────────────────────────────────
class SEBlock(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        mid = max(channels // reduction, 4)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc   = nn.Sequential(
            nn.Linear(channels, mid, bias=False), nn.ReLU(inplace=True),
            nn.Linear(mid, channels, bias=False), nn.Sigmoid())
    def forward(self, x):
        b, c = x.shape[:2]
        return x * self.fc(self.pool(x).view(b, c)).view(b, c, 1, 1)

class TumorClassifierConvNeXtTinySE(nn.Module):
    def __init__(self, n=2, r=16):
        super().__init__()
        bb            = torchvision.models.convnext_tiny(weights=None)
        self.features = bb.features
        self.se       = SEBlock(768, r)
        self.pool     = nn.AdaptiveAvgPool2d(1)
        self.head     = nn.Sequential(
            nn.Flatten(), nn.LayerNorm(768), nn.Dropout(0.6),
            nn.Linear(768, 256), nn.GELU(), nn.Dropout(0.5),
            nn.Linear(256, n))
    def forward(self, x):
        return self.head(self.pool(self.se(self.features(x))))

# ─── 4. PREPROCESSING ─────────────────────────────────────────────────────────
TFMS = T.Compose([T.ToTensor(),
                  T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])

def resize_pad(img, sz=256):
    w, h = img.size; r = min(sz/w, sz/h)
    nw, nh = int(w*r), int(h*r)
    img = img.resize((nw, nh), Image.BILINEAR)
    c = Image.new("RGB",(sz,sz),(0,0,0)); c.paste(img,((sz-nw)//2,(sz-nh)//2))
    return c

def preprocess(crop):
    img = ImageOps.autocontrast(crop.convert("L"), cutoff=1).convert("RGB")
    return TFMS(resize_pad(img)).unsqueeze(0)

def pil_bgr(img):
    return cv2.cvtColor(np.array(img.convert("RGB")), cv2.COLOR_RGB2BGR)

def crop_roi(img, box, pad=0.08):
    W, H = img.size; x1,y1,x2,y2 = box
    pw, ph = int((x2-x1)*pad), int((y2-y1)*pad)
    return img.crop((max(0,x1-pw),max(0,y1-ph),min(W,x2+pw),min(H,y2+ph)))

# ─── 5. MODEL LOADING ─────────────────────────────────────────────────────────
def build_detector():
    assert D2_OK,                         "detectron2 not installed"
    assert Path(DET_MODEL_PATH).exists(), f"Not found: {DET_MODEL_PATH}"
    cfg = get_cfg()
    cfg.merge_from_file(model_zoo.get_config_file(
        "COCO-Detection/faster_rcnn_R_50_FPN_3x.yaml"))
    cfg.MODEL.ROI_HEADS.NUM_CLASSES       = 1
    cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = DET_SCORE_THRESH
    cfg.MODEL.ROI_HEADS.NMS_THRESH_TEST   = NMS_THRESH
    cfg.MODEL.WEIGHTS                     = DET_MODEL_PATH
    cfg.MODEL.DEVICE                      = str(DEVICE)
    cfg.INPUT.MIN_SIZE_TEST               = 640
    cfg.INPUT.MAX_SIZE_TEST               = 1024
    cfg.TEST.DETECTIONS_PER_IMAGE         = 10
    cfg.MODEL.RPN.PRE_NMS_TOPK_TEST       = 1000
    cfg.MODEL.RPN.POST_NMS_TOPK_TEST      = 500
    cfg.MODEL.RPN.NMS_THRESH              = 0.5
    print(f"✅  Detector  (score_thresh={DET_SCORE_THRESH})")
    return DefaultPredictor(cfg)

def load_roi_model():
    assert Path(ROI_MODEL_PATH).exists(), f"Not found: {ROI_MODEL_PATH}"
    model = TumorClassifierConvNeXtTinySE(n=2, r=16)
    ckpt  = torch.load(ROI_MODEL_PATH, map_location=DEVICE, weights_only=False)
    state = ckpt.get("model_state_dict", ckpt) if isinstance(ckpt, dict) else ckpt
    model.load_state_dict(state, strict=True)
    print("✅  ROI classifier")
    return model.eval().to(DEVICE)

# ─── 6. INFERENCE HELPERS ─────────────────────────────────────────────────────
def detect(predictor, bgr):
    out = predictor(bgr)["instances"].to("cpu")
    if len(out) == 0: return []
    return [{"bbox_xyxy": out.pred_boxes.tensor.numpy()[i].astype(int).tolist(),
             "score"    : float(out.scores.numpy()[i])} for i in range(len(out))]

@torch.no_grad()
def classify(model, crop):
    p = F.softmax(model(preprocess(crop).to(DEVICE)), dim=1).squeeze(0).cpu().numpy()
    return float(p[0]), float(p[1])   # p_benign, p_malignant

def aggregate_rois(roi_results):
    n_mal  = sum(1 for r in roi_results if r["pm"] >= CLS_THRESHOLD)
    n_ben  = len(roi_results) - n_mal
    max_pm = max(r["pm"] for r in roi_results)
    max_pb = max(r["pb"] for r in roi_results)
    if AGG_MODE == "Any Malignant":
        label = "Malignant" if n_mal > 0 else "Benign"
    elif AGG_MODE == "Majority Vote":
        label = "Malignant" if n_mal >= n_ben else "Benign"
    else:
        label = "Malignant" if max_pm >= CLS_THRESHOLD else "Benign"
    return label, max_pm, max_pb

# ─── 7. LOAD & VALIDATE METADATA ─────────────────────────────────────────────
# ─── 7. LOAD & VALIDATE METADATA ─────────────────────────────────────────────
print("\n📋  Loading metadata …")
meta = pd.read_csv(META_CSV)
print(f"   Rows (raw): {len(meta)}  |  Columns: {list(meta.columns)}")

# ── Filter to test split only ──────────────────────────────────────────────────
if "split" in meta.columns:
    meta = meta[meta["split"] == "test"].reset_index(drop=True)
    print(f"   Rows after split='test' filter: {len(meta)}")
else:
    print("   [WARN] No 'split' column found — using all rows")

# ── Derive class_label from flag columns ──────────────────────────────────────
# Priority: malignant=1 → Malignant | benign=1 → Benign | else → Normal
def derive_label(row):
    if int(row.get("malignant", 0)) == 1:
        return "Malignant"
    elif int(row.get("benign", 0)) == 1:
        return "Benign"
    else:
        return "Normal"

meta["class_label"] = meta.apply(derive_label, axis=1)

meta["gt_s1"]   = (meta["class_label"] != "Normal").astype(int)
meta["gt_pipe"] = meta["class_label"]

print(f"\n   GT breakdown:")
for lbl in ["Normal", "Benign", "Malignant"]:
    print(f"     {lbl:<12}: {(meta['class_label']==lbl).sum():>5}")
print(f"     {'GT Tumor':<12}: {meta['gt_s1'].sum():>5}")
print(f"     {'Total':<12}: {len(meta):>5}")

# ── Resolve image paths ───────────────────────────────────────────────────────
def find_image(image_id: str):
    stem = Path(str(image_id)).stem
    for ext in IMG_EXTS:
        p = IMG_DIR / (stem + ext)
        if p.exists(): return str(p)
    p = IMG_DIR / str(image_id)
    if p.exists(): return str(p)
    return None

meta["img_path"] = meta["image_id"].astype(str).apply(find_image)
n_missing = meta["img_path"].isna().sum()
if n_missing:
    print(f"\n   [WARN] {n_missing} image(s) not found on disk — skipped")
meta = meta[meta["img_path"].notna()].reset_index(drop=True)
print(f"   Images matched: {len(meta)}")

# ─── 8. INFERENCE LOOP ────────────────────────────────────────────────────────
print("\n🔧  Loading models …")
detector  = build_detector()
roi_model = load_roi_model()

print("\n🔍  Running inference …")
t0   = time.time()
rows = []

for _, rec in tqdm(meta.iterrows(), total=len(meta), desc="BTXRD Test", unit="img"):
    try:
        pil = Image.open(rec["img_path"]).convert("RGB")
    except Exception as e:
        print(f"   [WARN] {rec['image_id']}: {e}"); continue

    # ── Stage 1 ───────────────────────────────────────────────────────────
    dets          = detect(detector, pil_bgr(pil))
    n_rois        = len(dets)
    det_pred      = 1 if n_rois > 0 else 0
    max_det_score = max((d["score"] for d in dets), default=0.0)

    # ── Stage 2 (only when tumor detected) ───────────────────────────────
    roi_results = []
    pipe_pred   = "Normal"
    max_pm      = 0.0
    max_pb      = 0.0

    if n_rois > 0:
        for det in dets:
            crop = crop_roi(pil, det["bbox_xyxy"], ROI_PADDING)
            pb, pm = classify(roi_model, crop)
            roi_results.append({"pb": pb, "pm": pm, "det_score": det["score"]})
        pipe_pred, max_pm, max_pb = aggregate_rois(roi_results)

    rows.append({
        "image_id"      : rec["image_id"],
        "center"        : rec.get("center", ""),
        "age"           : rec.get("age", ""),
        "gender"        : rec.get("gender", ""),
        "tumor_flag"    : rec.get("tumor", ""),
        "benign_flag"   : rec.get("benign", ""),
        "malignant_flag": rec.get("malignant", ""),
        # GT
        "gt_s1"         : int(rec["gt_s1"]),
        "gt_pipe"       : rec["gt_pipe"],
        # Stage-1
        "det_pred"      : det_pred,
        "n_rois"        : n_rois,
        "max_det_score" : max_det_score,
        # Stage-2
        "pipe_pred"     : pipe_pred,
        "max_pm"        : max_pm,
        "max_pb"        : max_pb,
        # Correctness
        "s1_correct"    : int(det_pred == int(rec["gt_s1"])),
        "pipe_correct"  : int(pipe_pred == rec["gt_pipe"]),
    })

elapsed = time.time() - t0
df = pd.DataFrame(rows)
df.to_csv(OUTPUT_DIR / "per_image_results.csv", index=False)
print(f"✅  Done: {len(df)} images in {elapsed:.1f}s  "
      f"({elapsed/len(df)*1000:.1f} ms/img)")

# ─── 9. METRICS ───────────────────────────────────────────────────────────────
# ─── 9. METRICS ───────────────────────────────────────────────────────────────
BG  = "#ffffff"
BG2 = "#f7f9fc"   # subtle panel fill
FG  = "#1a1a2e"
GRID_COL = "#e2e8f0"

plt.rcParams.update({
    "font.family"      : "DejaVu Sans",
    "axes.spines.top"  : False,
    "axes.spines.right": False,
    "axes.spines.left" : True,
    "axes.spines.bottom": True,
    "axes.edgecolor"   : "#cbd5e1",
    "axes.linewidth"   : 0.8,
    "xtick.direction"  : "out",
    "ytick.direction"  : "out",
    "xtick.major.size" : 3.5,
    "ytick.major.size" : 3.5,
    "xtick.color"      : "#475569",
    "ytick.color"      : "#475569",
    "figure.facecolor" : BG,
    "axes.facecolor"   : BG2,
    "grid.color"       : GRID_COL,
    "grid.linewidth"   : 0.6,
})

def compute_metrics(y_true, y_pred, y_score, title, names=None):
    print(f"\n{'▓'*62}\n  {title}\n{'▓'*62}")
    acc   = accuracy_score(y_true, y_pred)
    prec  = precision_score(y_true, y_pred, zero_division=0)
    rec   = recall_score(y_true, y_pred, zero_division=0)
    spec  = recall_score(y_true, y_pred, pos_label=0, zero_division=0)
    f1    = f1_score(y_true, y_pred, zero_division=0)
    mcc   = matthews_corrcoef(y_true, y_pred)
    kappa = cohen_kappa_score(y_true, y_pred)
    tn,fp,fn,tp = confusion_matrix(y_true, y_pred).ravel()
    npv   = tn/(tn+fn) if (tn+fn) else float("nan")
    try:
        auc_roc = roc_auc_score(y_true, y_score)
        auc_pr  = average_precision_score(y_true, y_score)
    except Exception:
        auc_roc = auc_pr = float("nan")
    m = dict(Accuracy=acc, Precision=prec, Recall_Sensitivity=rec,
             Specificity=spec, NPV=npv, F1=f1,
             AUC_ROC=auc_roc, AUC_PR=auc_pr, MCC=mcc, Kappa=kappa,
             TP=int(tp), TN=int(tn), FP=int(fp), FN=int(fn))
    for k, v in m.items():
        print(f"    {k:<28} {v:.4f}" if isinstance(v,float) else
              f"    {k:<28} {v}")
    names = names or ["Negative","Positive"]
    print(f"\n{classification_report(y_true, y_pred, target_names=names, zero_division=0)}")
    return m

# ── Level A: Stage-1 binary ───────────────────────────────────────────────────
m_A = compute_metrics(
    df["gt_s1"].values, df["det_pred"].values, df["max_det_score"].values,
    "LEVEL A — Stage-1: Normal (0) vs Tumor Detected (1)",
    names=["Normal","Tumor"])

# ── Level B: Full pipeline binary ────────────────────────────────────────────
y_true_pipe  = (df["gt_pipe"] != "Normal").astype(int).values
y_pred_pipe  = (df["pipe_pred"] != "Normal").astype(int).values
y_score_pipe = np.where(df["det_pred"]==1, df["max_pm"].values, 0.0)
m_B = compute_metrics(
    y_true_pipe, y_pred_pipe, y_score_pipe,
    "LEVEL B — Full Pipeline: Normal (0) vs Abnormal (1)",
    names=["Normal","Abnormal"])

# ── Level C: Stage-2 Benign vs Malignant on TP images ────────────────────────
print(f"\n{'▓'*62}")
print("  LEVEL C — Stage-2: Benign vs Malignant  (TP images only)")
print(f"{'▓'*62}")
df_tp = df[(df["det_pred"]==1) & (df["gt_s1"]==1)]
print(f"\n  TP images (det fired & GT=Tumor) : {len(df_tp)}")
print(f"    GT Benign    : {(df_tp['gt_pipe']=='Benign').sum()}")
print(f"    GT Malignant : {(df_tp['gt_pipe']=='Malignant').sum()}")

m_C = None
if len(df_tp) >= 4 and df_tp["gt_pipe"].nunique() == 2:
    y_tc = (df_tp["gt_pipe"]=="Malignant").astype(int).values
    y_pc = (df_tp["pipe_pred"]=="Malignant").astype(int).values
    y_sc = df_tp["max_pm"].values
    m_C  = compute_metrics(y_tc, y_pc, y_sc,
                           "Stage-2 on TP-Detected Images (Benign=0, Malignant=1)",
                           names=["Benign","Malignant"])
else:
    print("  [SKIP] Need both Benign & Malignant TP images.")

# ── Level D: Per-class breakdown ──────────────────────────────────────────────
print(f"\n{'▓'*62}\n  LEVEL D — Per-Class Breakdown\n{'▓'*62}")
for lbl in ["Normal","Benign","Malignant"]:
    sub = df[df["gt_pipe"]==lbl]
    if len(sub)==0: continue
    cor = (sub["pipe_pred"]==lbl).sum()
    print(f"\n  GT={lbl}  (n={len(sub)})")
    print(f"    Correct: {cor}/{len(sub)}  ({cor/len(sub)*100:.1f}%)")
    for p, c in sub["pipe_pred"].value_counts().items():
        mark = "✅" if p==lbl else "❌"
        print(f"    {mark}  Pred={p:<12}: {c:>5}  ({c/len(sub)*100:.1f}%)")

# ── Level E: 3-class confusion ────────────────────────────────────────────────
print(f"\n{'▓'*62}\n  LEVEL E — 3-Class Confusion Matrix\n{'▓'*62}")
label_order = ["Normal","Benign","Malignant"]
cm3    = confusion_matrix(df["gt_pipe"], df["pipe_pred"], labels=label_order)
cm3_df = pd.DataFrame(cm3,
    index  =[f"GT:{l}"   for l in label_order],
    columns=[f"Pred:{l}" for l in label_order])
print(f"\n{cm3_df.to_string()}")
print(f"\n{classification_report(df['gt_pipe'], df['pipe_pred'], zero_division=0)}")

# ── Bone-type subgroup ────────────────────────────────────────────────────────
bone_cols = ["humerus","ulna","radius","femur","tibia","fibula","foot",
             "hip bone","ankle-joint","knee-joint","hip-joint",
             "wrist-joint","elbow-joint","shoulder-joint"]
avail_bones = [c for c in bone_cols if c in meta.columns]
sub_rows = []
if avail_bones:
    print(f"\n{'▓'*62}\n  BONE-TYPE SUBGROUP — Stage-1 Accuracy\n{'▓'*62}")
    id_col = meta["image_id"].astype(str)
    df_id  = df["image_id"].astype(str)
    for bc in avail_bones:
        ids = meta.loc[meta[bc]==1, "image_id"].astype(str).tolist()
        sub = df[df_id.isin(ids)]
        if len(sub) < 5: continue
        acc = accuracy_score(sub["gt_s1"], sub["det_pred"])
        f1v = f1_score(sub["gt_s1"], sub["det_pred"], zero_division=0)
        sub_rows.append({"bone":bc, "n":len(sub), "acc":acc, "f1":f1v})
    if sub_rows:
        sub_df = pd.DataFrame(sub_rows).sort_values("acc",ascending=False)
        print(f"\n  {'Bone':<22} {'N':>5}  {'Acc':>7}  {'F1':>7}")
        print(f"  {'─'*44}")
        for _, r in sub_df.iterrows():
            print(f"  {r['bone']:<22} {int(r['n']):>5}  {r['acc']:>7.4f}  {r['f1']:>7.4f}")
        sub_df.to_csv(OUTPUT_DIR/"subgroup_bone_type.csv", index=False)

# ─── 10. PLOTS ────────────────────────────────────────────────────────────────
sns.set_theme(style="dark", font="monospace")

def sa(ax, title=""):
    ax.set_facecolor(BG2)
    ax.tick_params(colors="#475569", labelsize=9)
    ax.spines["left"].set_color("#cbd5e1")
    ax.spines["bottom"].set_color("#cbd5e1")
    ax.xaxis.label.set_color("#334155")
    ax.yaxis.label.set_color("#334155")
    ax.xaxis.label.set_fontsize(9)
    ax.yaxis.label.set_fontsize(9)
    ax.yaxis.grid(True, color=GRID_COL, linewidth=0.6, zorder=0)
    ax.set_axisbelow(True)
    if title:
        ax.set_title(title, fontsize=10, fontweight="bold",
                     color=FG, pad=7, loc="left")

PCOLS = {"Normal": "#22c55e", "Benign": "#3b82f6", "Malignant": "#ef4444"}
LCOLS = {"Stage-1": "#3b82f6", "Full Pipeline": "#8b5cf6", "Stage-2 B vs M": "#ef4444"}
# ── Plot 1: Confusion matrices ────────────────────────────────────────────────
# ── Plot 1: Confusion matrices ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5.2), facecolor=BG)
fig.suptitle("BoneScan AI — Confusion Matrices",
             color=FG, fontsize=13, fontweight="bold", x=0.02, ha="left", y=1.01)

CMAPS = ["Blues", "Purples", "Oranges"]
for ax in axes:
    ax.set_facecolor(BG2)

cm_s1 = confusion_matrix(df["gt_s1"], df["det_pred"])
ConfusionMatrixDisplay(cm_s1, display_labels=["Normal", "Tumor Det."]).plot(
    ax=axes[0], colorbar=False, cmap="Blues")
sa(axes[0], f"Stage-1   Acc={m_A['Accuracy']:.3f}  F1={m_A['F1']:.3f}")
for t in axes[0].texts: t.set_color(FG); t.set_fontsize(13); t.set_fontweight("bold")

cm_pipe = confusion_matrix(y_true_pipe, y_pred_pipe)
ConfusionMatrixDisplay(cm_pipe, display_labels=["Normal", "Abnormal"]).plot(
    ax=axes[1], colorbar=False, cmap="Purples")
sa(axes[1], f"Full Pipeline   Acc={m_B['Accuracy']:.3f}  F1={m_B['F1']:.3f}")
for t in axes[1].texts: t.set_color(FG); t.set_fontsize(13); t.set_fontweight("bold")

sns.heatmap(cm3, annot=True, fmt="d", cmap="YlOrRd",
            xticklabels=[f"Pred:{l}" for l in label_order],
            yticklabels=[f"GT:{l}"   for l in label_order],
            ax=axes[2], cbar=False,
            annot_kws={"color": FG, "fontsize": 12, "fontweight": "bold"},
            linewidths=0.5, linecolor=GRID_COL)
axes[2].tick_params(colors="#475569", labelsize=8.5)
sa(axes[2], "3-Class: Normal / Benign / Malignant")

plt.tight_layout(pad=1.5)
fig.savefig(OUTPUT_DIR/"confusion_matrices.png", dpi=180,
            bbox_inches="tight", facecolor=BG)
plt.close(fig); print("🖼  confusion_matrices.png")
# ── Plot 2: ROC + PR curves ───────────────────────────────────────────────────
# ── Plot 2: ROC + PR curves ───────────────────────────────────────────────────
n_plots = 3 if m_C else 2
fig, axes_grid = plt.subplots(2, n_plots, figsize=(6.5*n_plots, 10), facecolor=BG)
fig.suptitle("ROC and Precision-Recall Curves",
             color=FG, fontsize=13, fontweight="bold", x=0.02, ha="left", y=1.01)

CURVE_COLS = ["#3b82f6", "#8b5cf6", "#ef4444"]
plot_specs = [
    (df["gt_s1"].values, df["max_det_score"].values, m_A, "Stage-1 Detection"),
    (y_true_pipe, y_score_pipe, m_B, "Full Pipeline"),
]
if m_C:
    plot_specs.append((
        (df_tp["gt_pipe"]=="Malignant").astype(int).values,
        df_tp["max_pm"].values, m_C, "Stage-2 B vs M"))

for ci, (y_t, y_s, m, name) in enumerate(plot_specs):
    col  = CURVE_COLS[ci]
    ax_r = axes_grid[0][ci]
    ax_p = axes_grid[1][ci]

    if not np.isnan(m["AUC_ROC"]):
        fpr, tpr, _ = roc_curve(y_t, y_s)
        ax_r.plot(fpr, tpr, color=col, lw=2.2, label=f"AUC = {m['AUC_ROC']:.4f}", zorder=3)
        ax_r.fill_between(fpr, tpr, alpha=0.10, color=col)
    ax_r.plot([0,1],[0,1], "--", color="#94a3b8", lw=1.2, label="Random", zorder=2)
    ax_r.set_xlabel("False Positive Rate")
    ax_r.set_ylabel("True Positive Rate")
    ax_r.legend(frameon=True, facecolor=BG, edgecolor=GRID_COL,
                labelcolor=FG, fontsize=9, loc="lower right")
    sa(ax_r, f"ROC — {name}")

    if not np.isnan(m["AUC_PR"]):
        pr, rc, _ = precision_recall_curve(y_t, y_s)
        ax_p.plot(rc, pr, color=col, lw=2.2, label=f"AP = {m['AUC_PR']:.4f}", zorder=3)
        ax_p.fill_between(rc, pr, alpha=0.10, color=col)
    ax_p.axhline(y_t.mean(), color="#94a3b8", lw=1.2, linestyle="--",
                 label=f"Baseline = {y_t.mean():.2f}")
    ax_p.set_xlabel("Recall")
    ax_p.set_ylabel("Precision")
    ax_p.legend(frameon=True, facecolor=BG, edgecolor=GRID_COL,
                labelcolor=FG, fontsize=9, loc="upper right")
    sa(ax_p, f"PR Curve — {name}")

plt.tight_layout(pad=1.8)
fig.savefig(OUTPUT_DIR/"roc_pr_curves.png", dpi=180,
            bbox_inches="tight", facecolor=BG)
plt.close(fig); print("🖼  roc_pr_curves.png")
# ── Plot 3: Score distributions ───────────────────────────────────────────────
# ── Plot 3: Score distributions ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4.8), facecolor=BG)
fig.suptitle("Score Distributions by GT Class",
             color=FG, fontsize=13, fontweight="bold", x=0.02, ha="left")

for lbl in ["Normal", "Benign", "Malignant"]:
    sub = df[df["gt_pipe"] == lbl]
    if len(sub) == 0: continue
    axes[0].hist(sub["max_det_score"], bins=40, alpha=0.65,
                 color=PCOLS[lbl], label=f"GT={lbl}  (n={len(sub)})",
                 edgecolor="white", linewidth=0.3, zorder=3)
axes[0].axvline(DET_SCORE_THRESH, color="#1e293b", lw=1.6, linestyle="--",
                label=f"Threshold = {DET_SCORE_THRESH}", zorder=4)
axes[0].set_xlabel("Max Detection Score")
axes[0].set_ylabel("Count")
axes[0].legend(frameon=True, facecolor=BG, edgecolor=GRID_COL,
               labelcolor=FG, fontsize=9)
sa(axes[0], "Stage-1 Detection Score")

df_det = df[df["det_pred"] == 1]
for lbl in ["Benign", "Malignant"]:
    sub = df_det[df_det["gt_pipe"] == lbl]
    if len(sub) == 0: continue
    axes[1].hist(sub["max_pm"], bins=30, alpha=0.65,
                 color=PCOLS[lbl], label=f"GT={lbl}  (n={len(sub)})",
                 edgecolor="white", linewidth=0.3, zorder=3)
axes[1].axvline(CLS_THRESHOLD, color="#1e293b", lw=1.6, linestyle="--",
                label=f"Threshold = {CLS_THRESHOLD}", zorder=4)
axes[1].set_xlabel("Max P(Malignant)")
axes[1].set_ylabel("Count")
axes[1].legend(frameon=True, facecolor=BG, edgecolor=GRID_COL,
               labelcolor=FG, fontsize=9)
sa(axes[1], "Stage-2 P(Malignant) — Detected Images")

plt.tight_layout(pad=1.5)
fig.savefig(OUTPUT_DIR/"score_distributions.png", dpi=180,
            bbox_inches="tight", facecolor=BG)
plt.close(fig); print("🖼  score_distributions.png")
# ── Plot 4: Metrics bar ───────────────────────────────────────────────────────
# ── Plot 4: Metrics bar ───────────────────────────────────────────────────────
KNAMES = ["Accuracy", "Precision", "Recall_Sensitivity",
          "Specificity", "F1", "AUC_ROC", "AUC_PR", "MCC"]
all_m = [("Stage-1", m_A, "#3b82f6"),
         ("Full Pipeline", m_B, "#8b5cf6")]
if m_C: all_m.append(("Stage-2 B vs M", m_C, "#ef4444"))

fig, ax = plt.subplots(figsize=(14, 5.2), facecolor=BG)
x = np.arange(len(KNAMES)); n = len(all_m); w = 0.68 / n

for i, (name, m, col) in enumerate(all_m):
    offset = (i - n / 2 + 0.5) * w
    bars = ax.bar(x + offset, [m[k] for k in KNAMES], w,
                  label=name, color=col, alpha=0.88,
                  edgecolor="white", linewidth=0.5, zorder=3)
    for bar in bars:
        h = bar.get_height()
        if not np.isnan(h) and h > 0.04:
            ax.text(bar.get_x() + bar.get_width() / 2, h + 0.012,
                    f"{h:.3f}", ha="center", va="bottom",
                    color=FG, fontsize=7, fontweight="bold")

ax.set_xticks(x)
ax.set_xticklabels(KNAMES, rotation=28, ha="right", fontsize=9.5, color="#334155")
ax.set_ylim(0, 1.20)
ax.set_ylabel("Score", fontsize=10, color="#334155")
ax.legend(frameon=True, facecolor=BG, edgecolor=GRID_COL,
          labelcolor=FG, fontsize=9.5, loc="upper right")
sa(ax, f"Performance Summary  [det_thr={DET_SCORE_THRESH}  cls_thr={CLS_THRESHOLD}]")

plt.tight_layout(pad=1.5)
fig.savefig(OUTPUT_DIR/"metrics_bar.png", dpi=180,
            bbox_inches="tight", facecolor=BG)
plt.close(fig); print("🖼  metrics_bar.png")
# ── Plot 5: Per-class prediction breakdown ─────────────────────────────────────
# ── Plot 5: Per-class prediction breakdown ────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5.2), facecolor=BG)
fig.suptitle("Prediction Breakdown per GT Class",
             color=FG, fontsize=13, fontweight="bold", x=0.02, ha="left")

for ax, gt_lbl in zip(axes, ["Normal", "Benign", "Malignant"]):
    sub = df[df["gt_pipe"] == gt_lbl]
    if len(sub) == 0:
        sa(ax, f"GT={gt_lbl}  (n=0)"); continue
    bottom = 0.0
    for pred_lbl in ["Normal", "Benign", "Malignant"]:
        cnt = (sub["pipe_pred"] == pred_lbl).sum()
        if cnt == 0: continue
        pct = cnt / len(sub) * 100
        ax.bar(gt_lbl, pct, bottom=bottom,
               color=PCOLS[pred_lbl], alpha=0.88,
               edgecolor="white", linewidth=0.6, zorder=3)
        if pct > 4:
            ax.text(0, bottom + pct / 2,
                    f"{pred_lbl}\n{pct:.1f}%",
                    ha="center", va="center",
                    color="white" if pred_lbl == "Malignant" else FG,
                    fontsize=9, fontweight="bold")
        bottom += pct
    ax.set_ylim(0, 118)
    ax.set_ylabel("% of Images", fontsize=9)
    sa(ax, f"GT = {gt_lbl}   (n={len(sub)})")

plt.tight_layout(pad=1.5)
fig.savefig(OUTPUT_DIR/"prediction_breakdown.png", dpi=180,
            bbox_inches="tight", facecolor=BG)
plt.close(fig); print("🖼  prediction_breakdown.png")
# ── Plot 6: Bone-type subgroup bar ────────────────────────────────────────────
# ── Plot 6: Bone-type subgroup bar ────────────────────────────────────────────
bone_csv = OUTPUT_DIR / "subgroup_bone_type.csv"
if bone_csv.exists():
    sub_df = pd.read_csv(bone_csv)
    if len(sub_df) >= 2:
        fig, ax = plt.subplots(figsize=(13, 5), facecolor=BG)
        xb = np.arange(len(sub_df)); wb = 0.35
        ax.bar(xb - wb/2, sub_df["acc"], wb, color="#3b82f6", alpha=0.85,
               label="Accuracy", edgecolor="white", linewidth=0.5, zorder=3)
        ax.bar(xb + wb/2, sub_df["f1"],  wb, color="#8b5cf6", alpha=0.85,
               label="F1 Score",  edgecolor="white", linewidth=0.5, zorder=3)
        ax.set_xticks(xb)
        ax.set_xticklabels(sub_df["bone"], rotation=35, ha="right",
                           fontsize=9, color="#334155")
        ax.set_ylim(0, 1.15)
        ax.set_ylabel("Score", fontsize=10)
        ax.legend(frameon=True, facecolor=BG, edgecolor=GRID_COL,
                  labelcolor=FG, fontsize=9.5)
        sa(ax, "Stage-1 Detection Accuracy by Bone Type")
        plt.tight_layout(pad=1.5)
        fig.savefig(OUTPUT_DIR/"subgroup_bone_type.png", dpi=180,
                    bbox_inches="tight", facecolor=BG)
        plt.close(fig); print("🖼  subgroup_bone_type.png")
# ─── 11. SAVE REPORT ──────────────────────────────────────────────────────────
KNAMES_R = ["Accuracy","Precision","Recall_Sensitivity","Specificity",
             "F1","AUC_ROC","AUC_PR","MCC","Kappa"]

lines = [
    "="*64,
    "  BoneScan AI — BTXRD Test Split Evaluation",
    f"  det_thr={DET_SCORE_THRESH}  cls_thr={CLS_THRESHOLD}  agg={AGG_MODE}",
    "="*64, "",
    f"  GT Normal    : {(df['gt_pipe']=='Normal').sum()}",
    f"  GT Benign    : {(df['gt_pipe']=='Benign').sum()}",
    f"  GT Malignant : {(df['gt_pipe']=='Malignant').sum()}",
    f"  Total        : {len(df)}", "",
    "─"*64, "  LEVEL A — Stage-1: Normal vs Tumor", "─"*64,
]
for k in KNAMES_R: lines.append(f"    {k:<28} {m_A[k]:.4f}")
lines += [f"    TP/TN/FP/FN = {m_A['TP']}/{m_A['TN']}/{m_A['FP']}/{m_A['FN']}",
          "", "─"*64, "  LEVEL B — Full Pipeline: Normal vs Abnormal", "─"*64]
for k in KNAMES_R: lines.append(f"    {k:<28} {m_B[k]:.4f}")
lines += [f"    TP/TN/FP/FN = {m_B['TP']}/{m_B['TN']}/{m_B['FP']}/{m_B['FN']}", ""]

if m_C:
    lines += ["─"*64, "  LEVEL C — Stage-2: Benign vs Malignant (TP only)", "─"*64]
    for k in KNAMES_R: lines.append(f"    {k:<28} {m_C[k]:.4f}")
    lines += [f"    TP/TN/FP/FN = {m_C['TP']}/{m_C['TN']}/{m_C['FP']}/{m_C['FN']}", ""]

lines += ["─"*64, "  LEVEL E — 3-Class Confusion", "─"*64,
          cm3_df.to_string(), "",
          "="*64,
          f"  Inference: {elapsed:.1f}s  ({elapsed/len(df)*1000:.1f} ms/img)",
          "="*64]

report = "\n".join(lines)
print("\n" + report)
(OUTPUT_DIR/"evaluation_report.txt").write_text(report)

json_out = {
    "config"           : {"det_thr":DET_SCORE_THRESH, "cls_thr":CLS_THRESHOLD, "agg":AGG_MODE},
    "stage1_detection" : {k:float(v) if isinstance(v,float) else v for k,v in m_A.items()},
    "full_pipeline"    : {k:float(v) if isinstance(v,float) else v for k,v in m_B.items()},
    "stage2_b_vs_m"    : ({k:float(v) if isinstance(v,float) else v
                           for k,v in m_C.items()} if m_C else None),
}
(OUTPUT_DIR/"metrics.json").write_text(json.dumps(json_out, indent=2))

# ─── 12. OUTPUT LISTING ───────────────────────────────────────────────────────
print("\n" + "═"*60)
print(f"  📦  Outputs → {OUTPUT_DIR}")
print("═"*60)
for f in sorted(OUTPUT_DIR.iterdir()):
    print(f"    {f.name:<42}  {f.stat().st_size/1024:>7.1f} KB")
print("═"*60)
print("✅  Evaluation complete.")

✅  detectron2 ready
✅  Device : cuda

📋  Loading metadata …
   Rows (raw): 564  |  Columns: ['image_id', 'center', 'age', 'gender', 'hand', 'ulna', 'radius', 'humerus', 'foot', 'tibia', 'fibula', 'femur', 'hip bone', 'ankle-joint', 'knee-joint', 'hip-joint', 'wrist-joint', 'elbow-joint', 'shoulder-joint', 'tumor', 'benign', 'malignant', 'osteochondroma', 'multiple osteochondromas', 'simple bone cyst', 'giant cell tumor', 'osteofibroma', 'synovial osteochondroma', 'other bt', 'osteosarcoma', 'other mt', 'upper limb', 'lower limb', 'pelvis', 'frontal', 'lateral', 'oblique', 'patient_id', 'split']
   Rows after split='test' filter: 564

   GT breakdown:
     Normal      :   278
     Benign      :   239
     Malignant   :    47
     GT Tumor    :   286
     Total       :   564
   Images matched: 564

🔧  Loading models …
✅  Detector  (score_thresh=0.15)
✅  ROI classifier

🔍  Running inference …


BTXRD Test: 100%|██████████| 564/564 [01:24<00:00,  6.64img/s]


✅  Done: 564 images in 85.0s  (150.6 ms/img)

▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓
  LEVEL A — Stage-1: Normal (0) vs Tumor Detected (1)
▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓
    Accuracy                     0.8333
    Precision                    0.8038
    Recall_Sensitivity           0.8881
    Specificity                  0.7770
    NPV                          0.8710
    F1                           0.8439
    AUC_ROC                      0.9061
    AUC_PR                       0.9141
    MCC                          0.6699
    Kappa                        0.6661
    TP                           254
    TN                           216
    FP                           62
    FN                           32

              precision    recall  f1-score   support

      Normal       0.87      0.78      0.82       278
       Tumor       0.80      0.89      0.84       286

    accuracy                           0.83       564
   macro 

In [3]:
import shutil

# Source folder
folder_path = "/kaggle/working/"

# Output zip path (without .zip)
zip_path = "/kaggle/working/full pipeline on test set"

# Create zip
shutil.make_archive(zip_path, 'zip', folder_path)

print("✅ Zipped successfully!")
print("Zip file location:", zip_path + ".zip")

✅ Zipped successfully!
Zip file location: /kaggle/working/full pipeline on test set.zip
